In [1]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
from collections import defaultdict
import pandas as pd
import os

In [2]:

def get_cell_colors(wb,cns,sheetname):
    """
    统计Excel中所有内容为"dock"的单元格背景色
    
    参数:
        file_path: Excel文件路径
        sheet_name: 工作表名称(默认为"雪夜Q吧唧")
        
    返回:
        字典: {背景色RGB值: 出现次数}
    """
    # 加载工作簿和工作表
    sheet=wb[sheetname] #遍历sheets并打开对应sheet
    colors = []

    for cn in cns:
        # 遍历所有单元格
        for row in sheet.iter_rows():
            for cell in row:
                # 检查单元格值是否为"dock"
                if str(cell.value) == str(cn):
                    fill = cell.fill
                    

                    # 获取前景色(通常代表填充色)
                    color = fill.fgColor.rgb if fill.fgColor else None                        
                    
                    # 统计颜色
                    if color:
                        colors.append([cn,str(color),1])
                    else:
                        # 无填充色的情况
                        colors.append([cn,'无填充色',1])
    colors=pd.DataFrame(colors,columns=['cn','颜色','数量'])
    #重复项合并求和
    colors=colors.groupby(['cn','颜色'],as_index=False).sum().reindex(columns=['cn','颜色','数量'])
    #合并盲抽/非盲抽部分
    return colors


In [7]:
def save_to_excel(df, obj_path, sheet_name):
    """
    将DataFrame保存到Excel文件
    - 如果文件不存在，创建新文件
    - 如果文件存在，追加新Sheet（如果Sheet已存在则覆盖）
    
    参数:
        df: 要保存的DataFrame
        file_path: Excel文件路径
        sheet_name: 要写入的Sheet名称
        index: 是否保留索引（默认False）
    """
    # 检查文件是否存在
    #新建系列文件夹
    if not os.path.exists(obj_path):
        os.makedirs(obj_path,exist_ok=True)
    #按照sheetname分别存储
    df.to_excel(os.path.join(obj_path,f'{sheet_name}.xlsx'),sheet_name=sheet_name,header=True,index=False)
    
    print(f"创建新文件 {obj_path} 并写入Sheet '{sheet_name}'")
    

In [4]:
def get_cns_schedule(name,file_path,wb,sheetname):
    print("{}/{} processing.\n".format(name,sheetname))
    sheet=wb[sheetname] #遍历sheets并打开对应sheet
    max_col = sheet.max_column  #获取最大列数
    '''
    sheetname=pd.read_excel(file_path,header=None,nrows=1,sheet_name=sheetname)
    sheetname=sheetname.iloc[0,0].split('【')[-1].split('】')[0]
    '''
    table = pd.read_excel(
    file_path,
    skiprows=1,        # 跳过第一行
    usecols=range(1,max_col),     # 从 B 列开始读取（跳过第一列）
    sheet_name=sheetname
    ).drop(0)   #读取表格
    oall=table.count().sum()

    cns=set()
    for index,content in table.items():
        goodsname=index
        for cn in content.values:
            if pd.isna(cn):
                continue
            else:
                cns.add(cn)
    return cns


In [ ]:
name='2月韩国M2快闪（已到齐）'

#设置囤货
storages=['萌','dock','黄油']
storage=storages[1]

file_path = rf"./旧排发表/{storage}/{name}.xlsx"

filename=os.path.basename(file_path).split('.xlsx')[0]  #获取文件名/系列
obj_path=f'./colors/{name}'   #生成目标地址

mainmodes=['shopping','schedule']
mainmode=mainmodes[1]
shoppingsheets=["12.27大邱韩国30周年现货","单领","12月大邱30周年韩国凑满赠","3.8四宫格贴纸","3.18扫街58汇","64.12韩国30周年dg掉落"]
schedulesheets=["c站企鹅qqr","CRUX 饮料Q吧唧"]

wb=load_workbook(file_path)
cns=set()

for sheetname in wb.sheetnames:
    cns.clear()
    if mainmode=='shopping':
        if sheetname in schedulesheets:
            cns=get_cns_schedule(name,file_path,wb,sheetname)
            colors=get_cell_colors(wb,cns,sheetname)
            save_to_excel(colors,obj_path,sheetname)
        else:
            continue
    elif mainmode=='schedule':
        if sheetname in shoppingsheets:
            continue
        else:
            cns=get_cns_schedule(name,file_path,wb,sheetname)
            colors=get_cell_colors(wb,cns,sheetname)
            save_to_excel(colors,obj_path,sheetname)

    






2月韩国M2快闪（已到齐）/2月m2快闪 processing.

创建新文件 ./colors/2月韩国M2快闪（已到齐） 并写入Sheet '2月m2快闪'
2月韩国M2快闪（已到齐）/M2小卡打印 processing.

创建新文件 ./colors/2月韩国M2快闪（已到齐） 并写入Sheet 'M2小卡打印'
